## Quadrotor

Imports

In [ ]:
import importlib
import numpy as np

# Trajopt imports --> pip install -e ~/ACL/trajopt
import trajopt; importlib.reload(trajopt)
from trajopt.core.trajectory_analyzer import TrajectoryAnalyzer
np.random.seed(0)  # for reproducibility

setup problem and run SCP

In [ ]:
# create trajectroy analyzer object
mission = "mission.yaml"
model   = "trajopt/library/models/quadrotor_3dof.yaml"
method  = "method.yaml"

trajopt = TrajectoryAnalyzer(mission, model, method)

trajopt.solve()

In [ ]:
data = trajopt.analyze(analysis_type="standalone", animate=True)

# PSEUDOSPECTRAL

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Import the streamlined PS constraint building functions
from trajopt.library.methods.discretize import (
    compute_ps_differentiation_matrix,
    compute_ps_dynamics_and_jacobians
)
from trajopt.library.models.quadrotor_3dof import dynamics_jax
import trajopt.utils.tools as tools

# ----------------------------------------------------------------
# Pull SCP solution (uniform FOH grid)  
# ----------------------------------------------------------------
sol    = trajopt.solution
params = trajopt.problem.params
fcns   = trajopt.problem.fcns

t_opt  = np.asarray(sol['t_opt']).reshape(-1)   # (N+1,)
x_opt  = np.asarray(sol['x_opt'])                # (N+1, nx)
u_opt  = np.asarray(sol['u_opt'])                # (N,   nu)

t0     = t_opt[0]
tf     = t_opt[-1]
x0     = x_opt[0, :]

# ----------------------------------------------------------------
# Use streamlined PS constraint building functions
# ----------------------------------------------------------------
N_scp   = len(t_opt) - 1   # SCP resolution
N_col   = 20 * N_scp      # PS collocation points

# Get PS differentiation matrix using streamlined function
tau, etau, w, D = compute_ps_differentiation_matrix(N_col)

# Map tau in [-1,1] -> physical time
t_etau  = 0.5 * (tf - t0) * (etau) + 0.5 * (tf + t0)   # (N+1,) — includes t0
t_tau   = 0.5 * (tf - t0) * (tau) + 0.5 * (tf + t0)    # (N_col,) — collocation nodes only

# ----------------------------------------------------------------
# Interpolate u_opt (FOH) onto etau/tau time grid
# ----------------------------------------------------------------
def interp_u_foh(t_query, t_grid, u_grid):
    """First-order hold: linearly interpolate between u_grid[k] and u_grid[k+1]."""
    idx   = np.searchsorted(t_grid[:-1], t_query, side='right') - 1
    idx   = np.clip(idx, 0, len(u_grid) - 2)

    t_k   = t_grid[idx]
    t_kp  = t_grid[idx + 1]
    u_k   = u_grid[idx]
    u_kp  = u_grid[idx + 1]

    a = (t_kp - t_query) / (t_kp - t_k)   # weight on k
    b = (t_query - t_k)  / (t_kp - t_k)   # weight on k+1

    return a * u_k + b * u_kp

u_etau = np.array([interp_u_foh(t, t_opt, u_opt) for t in t_etau])  # (N+1, nu)
u_tau  = np.array([interp_u_foh(t, t_opt, u_opt) for t in t_tau ])  # (N_col, nu)

# ----------------------------------------------------------------
# Re-propagate dynamics with solve_ivp evaluated at etau nodes
# ----------------------------------------------------------------
def ode(t, x):
    u = interp_u_foh(t, t_opt, u_opt)
    return np.array(dynamics_jax(t, jnp.array(x), jnp.array(u), params, fcns))

ivp_sol = solve_ivp(
    ode,
    (t0, tf),
    x0,
    method       = "RK45",
    t_eval       = t_etau,
    dense_output = True,
    rtol         = 1e-10,
    atol         = 1e-12,
)
assert ivp_sol.success, f"IVP failed: {ivp_sol.message}"

X = ivp_sol.y.T   # (N+1, nx) — state at etau nodes

# ----------------------------------------------------------------
# Streamlined PS defect analysis
# ----------------------------------------------------------------
def analyze_ps_defects_streamlined(X_etau, u_tau, t_etau, t_tau, D, problem, method):
    """
    Streamlined PS defect analysis using constraint building infrastructure.
    """
    scale_ps = 2.0 / (t_etau[-1] - t_etau[0])  # Physical time scaling
    
    # LHS: pseudospectral derivative D @ X (scaled to physical time)
    X_dot_ps = scale_ps * (D @ X_etau)
    
    # Build z_ref and nu_ref in the expected format
    n_z = problem.index_map.n.z
    n_nu = problem.index_map.n.nu
    nx = X_etau.shape[1]
    N_col = len(t_tau)
    
    # Construct z_ref (includes state and possibly time)
    z_ref = np.zeros((N_col + 1, n_z))
    
    # Map physical states to z indices
    state_idx = problem.index_map.indices.z.state
    if state_idx is not None and len(state_idx) == nx:
        z_ref[:, state_idx] = X_etau
    else:
        # Fallback: assume states are first nx components
        z_ref[:, :nx] = X_etau
    
    # Add time if it's part of the state vector
    time_idx = problem.index_map.indices.z.time
    if time_idx is not None and len(time_idx) > 0:
        z_ref[:, time_idx] = t_etau.reshape(-1, 1)
    
    # Construct nu_ref (control inputs)
    nu_ref = np.zeros((N_col, n_nu))
    control_idx = problem.index_map.indices.nu.control
    if control_idx is not None:
        nu_ref[:, control_idx] = u_tau
    else:
        # Fallback: assume controls are first components
        nu_ref[:, :u_tau.shape[1]] = u_tau
    
    # Try streamlined dynamics computation
    try:
        f_ref_col, Ac_col, Bc_col = compute_ps_dynamics_and_jacobians(
            z_ref, nu_ref, problem, method
        )
        
        # Extract only the state derivative components
        if state_idx is not None:
            X_dot_f = f_ref_col[:, state_idx]
        else:
            X_dot_f = f_ref_col[:, :nx]
            
        method_used = "streamlined"
        
    except Exception as e:
        print(f"Streamlined approach failed: {e}")
        print("Falling back to manual dynamics computation...")
        
        # Fallback: manual computation
        X_tau = X_etau[1:, :]  # Collocation nodes (drop initial node)
        X_dot_f = np.array([
            np.array(dynamics_jax(t_tau[k], jnp.array(X_tau[k]), jnp.array(u_tau[k]), params, fcns))
            for k in range(N_col)
        ])
        method_used = "manual"
        Ac_col, Bc_col = None, None
    
    defect = X_dot_ps - X_dot_f
    
    return defect, X_dot_ps, X_dot_f, method_used, (Ac_col, Bc_col)

# Perform streamlined analysis
defect, X_dot_ps, X_dot_f, method_used, jacobians = analyze_ps_defects_streamlined(
    X, u_tau, t_etau, t_tau, D, trajopt.problem, trajopt.method
)

# ----------------------------------------------------------------
# Results display
# ----------------------------------------------------------------
state_labels = ["x", "y", "z", "vx", "vy", "vz"]

print(f"PS Defect Analysis Results:")
print(f"- Method: {method_used}")
print(f"- Collocation nodes: {N_col}")
print(f"- Time span: [{t0:.3f}, {tf:.3f}] s")
if jacobians[0] is not None:
    print(f"- Jacobians computed: Ac {jacobians[0].shape}, Bc {jacobians[1].shape}")
print()

print(f"{'State':<8}  {'Max |defect|':>14}  {'Mean |defect|':>14}")
print("-" * 42)
for i, label in enumerate(state_labels):
    print(f"{label:<8}  {np.max(np.abs(defect[:, i])):>14.6e}  {np.mean(np.abs(defect[:, i])):>14.6e}")
print(f"\n{'TOTAL':<8}  {np.max(np.abs(defect)):>14.6e}")

# Store for plotting
N = N_col  # For compatibility with plotting code

In [ ]:
# ----------------------------------------------------------------
# Plot
# ----------------------------------------------------------------
fig, axs = plt.subplots(2, 3, figsize=(14, 6))
axs = axs.flatten()

for i, label in enumerate(state_labels):
    axs[i].plot(t_tau, defect[:, i], marker="o", ms=3)
    axs[i].axhline(0, color="k", lw=0.8, ls="--")
    axs[i].set_title(f"Defect: $\\dot{{{label}}}$ — PS vs $f(x,u)$")
    axs[i].set_xlabel("time [s]")
    axs[i].set_ylabel("residual")
    axs[i].grid(True, alpha=0.3)

plt.suptitle("PS defect  $D\\,X - f(x,u)$  [re-propagated on fLGR grid, FOH]", fontsize=13)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------
# Plot propagated states at etau nodes
# ----------------------------------------------------------------
fig, axs = plt.subplots(2, 3, figsize=(14, 6))
axs = axs.flatten()

for i, label in enumerate(state_labels):
    axs[i].plot(t_etau, X[:, i],    lw=2,  label="IVP (re-propagated)")
    axs[i].plot(t_opt,  x_opt[:, i], lw=1.5, ls="--", marker="o", ms=3, label="SCP solution")
    axs[i].set_title(f"State: ${label}$")
    axs[i].set_xlabel("time [s]")
    axs[i].set_ylabel(label)
    axs[i].legend(fontsize=8)
    axs[i].grid(True, alpha=0.3)

plt.suptitle("Propagated states vs SCP solution", fontsize=13)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------
# Plot 3D trajectory
# ----------------------------------------------------------------
fig = plt.figure(figsize=(7, 6))
ax  = fig.add_subplot(111, projection='3d')

ax.plot(X[:, 0],      X[:, 1],      X[:, 2],      lw=2,        label="IVP (re-propagated)")
ax.plot(x_opt[:, 0],  x_opt[:, 1],  x_opt[:, 2],  lw=1.5, ls="--", marker="o", ms=3, label="SCP solution")

ax.scatter(*X[0,  :3], color="green", s=60, zorder=5, label="start")
ax.scatter(*X[-1, :3], color="red",   s=60, zorder=5, label="end")

ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_zlabel("z [m]")
ax.set_title("3D trajectory")
ax.legend()
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------
# Plot control inputs
# ----------------------------------------------------------------
fig, axs = plt.subplots(1, 3, figsize=(14, 4))

u_labels = ["Tx", "Ty", "Tz"]
for i, label in enumerate(u_labels):
    axs[i].plot(t_etau, u_etau[:, i], lw=2, label="FOH on etau")
    axs[i].plot(t_opt, u_opt[:, i], lw=1.5, ls="--", marker="o", ms=3, label="SCP u_opt")  # u_opt is (N+1,) for FOH
    axs[i].set_title(f"Control: ${label}$")
    axs[i].set_xlabel("time [s]")
    axs[i].set_ylabel(label)
    axs[i].legend(fontsize=8)
    axs[i].grid(True, alpha=0.3)

plt.suptitle("Control inputs — SCP vs FOH interpolation on fLGR grid", fontsize=13)
plt.tight_layout()
plt.show()

# ...existing code...

# ----------------------------------------------------------------
# Plot thrust magnitude (2-norm)
# ----------------------------------------------------------------
T_mag_etau = np.linalg.norm(u_etau, axis=1)   # (N+1,)
T_mag_opt  = np.linalg.norm(u_opt,  axis=1)   # (N+1,)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_etau, T_mag_etau, lw=2,        label="FOH on etau")
ax.plot(t_opt,  T_mag_opt,  lw=1.5, ls="--", marker="o", ms=3, label="SCP u_opt")
ax.set_title("Thrust magnitude $\\|u\\|_2$")
ax.set_xlabel("time [s]")
ax.set_ylabel("$\\|u\\|_2$  [N]")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()